<p class="hz-kicker">Week 3 · Self-paced lab</p>

# A rate simulator

Altamar's revenue manager wants to *use* the model, not just admire it: type in a booking, get a rate, and know when to trust it. In this lab you turn the week's linear model into a rate simulator, test it one input at a time, hunt down its worst misses, and write the note that goes with it.

<ul class="hz-lesson-meta"><li>≈ 2 hours</li><li>5 tasks</li><li>Runs in your browser</li><li>Graded quiz</li></ul>

**Recap of the live session**

- A model is a **recipe learned from past bookings**: it adds an intercept and one coefficient per input. Text columns become 0/1 columns, measured against a **reference** (Santiago, Standard, Direct, Leisure, January).
- We learn on 75% of the bookings and judge on the sealed 25% (**test set**). The linear model is typically off by about **USD 20 per night** (16% of the average rate), R² ≈ 0.74, far better than quoting each hotel's average.
- Its misses follow a **pattern**: one season shape for three hotels, and peaks it never reaches.

**Run this cell first. You don't need to read it.** It loads the tools and the helpers of the live session, plus `quote`, which prices one booking and prints its recipe.

In [1]:
# Setup: tools for today. Run this cell first; you don't need to read it.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

DATA = "https://raw.githubusercontent.com/hizocar/mba-business-analytics-1/main/docs/datasets/"
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
HOTELS = ["Santiago", "Viña del Mar", "Puerto Varas"]
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

# Every category of each text column. The FIRST one is the reference: its coefficient is 0,
# and the model measures the others against it.
LEVELS = {
    "hotel": ["Santiago", "Viña del Mar", "Puerto Varas"],
    "room_type": ["Standard", "Superior", "Suite"],
    "channel": ["Direct", "Online travel agency", "Corporate", "Travel agent"],
    "customer_type": ["Leisure", "Business", "Group"],
    "deposit": ["No deposit", "Non-refundable"],
    "country": ["Chile", "Argentina", "Brazil", "United States", "Europe", "Other", "Unknown"],
    "arrival_month": list(range(1, 13)),
}

def make_X(data, features):
    """Turn the chosen columns into numbers: one 0/1 column per category, minus the reference."""
    parts = []
    for col in features:
        if col in LEVELS:
            values = pd.Series(pd.Categorical(data[col], categories=LEVELS[col]), index=data.index)
            parts.append(pd.get_dummies(values, prefix=col, drop_first=True, dtype="int64"))
        else:
            parts.append(data[[col]])
    return pd.concat(parts, axis=1)

def scores(name, actual, predicted):
    """One row of a results table: the typical error in USD per night, and R²."""
    mae = mean_absolute_error(actual, predicted)
    return {"model": name, "MAE (USD/night)": round(mae, 1),
            "MAE as % of avg rate": f"{mae / actual.mean():.0%}", "R²": round(r2_score(actual, predicted), 3)}

# A booking described in plain words. Anything you don't set takes these values.
DEFAULTS = {"hotel": "Santiago", "arrival_month": 1, "room_type": "Standard", "channel": "Direct",
            "customer_type": "Leisure", "nights": 3, "weekend_nights": 0, "lead_time_days": 30, "adults": 2,
            "children": 0, "deposit": "No deposit", "repeat_guest": 0, "previous_cancellations": 0,
            "special_requests": 0, "parking": 0, "country": "Chile"}

def contributions(model, features, booking):
    """What each input adds to the predicted rate of one booking, in USD per night."""
    row = pd.DataFrame([{**DEFAULTS, **booking}])
    x = make_X(row, features).iloc[0]
    parts = x * pd.Series(model.coef_, index=x.index)
    owner = [next(f for f in features if c == f or c.startswith(f + "_")) for c in x.index]
    return parts.groupby(owner, sort=False).sum().reindex(features), {**DEFAULTS, **booking}

def quote(model, features, show=True, **booking):
    """Price one booking. With show=True, print the recipe: intercept + one line per input."""
    parts, full = contributions(model, features, booking)
    total = model.intercept_ + parts.sum()
    if show:
        keep = [f for f in features if f in booking or abs(parts[f]) >= 0.05]
        table = pd.DataFrame({"value": [full[f] for f in keep], "adds (USD/night)": parts[keep].round(1)}, index=keep)
        table.loc["start (intercept)"] = ["Santiago · Jan · Standard · Direct", round(model.intercept_, 1)]
        table = table.loc[["start (intercept)"] + keep]
        print(table.to_string())
        print(f"{'PREDICTED RATE':<24} USD {total:.1f} per night   (inputs left at their reference add 0)")
    return total

print("Ready: tools loaded.")

Ready: tools loaded.


Load the clean bookings, split them as in class, and fit the full linear model from the live session (step 06). Check the test error: it should match the live session.

In [2]:
df = pd.read_csv(DATA + "altamar_bookings_clean.csv")
df["arrival_month"] = pd.to_datetime(df["arrival_date"]).dt.month
train, test = train_test_split(df, test_size=0.25, random_state=42)
FEATURES = ["hotel", "arrival_month", "room_type", "channel", "customer_type", "nights", "weekend_nights",
            "lead_time_days", "adults", "children", "deposit", "repeat_guest", "previous_cancellations",
            "special_requests", "parking", "country"]
X_train, X_test = make_X(train, FEATURES), make_X(test, FEATURES)
y_train, y_test = train["adr"], test["adr"]
model = LinearRegression().fit(X_train, y_train)
print(pd.DataFrame([scores("Full linear model, test set", y_test, model.predict(X_test))]).to_string(index=False))

                      model  MAE (USD/night) MAE as % of avg rate    R²
Full linear model, test set             20.5                  16% 0.738


## <span class="hz-step-n">01</span> Quote a booking

A guest asks for a price. Describe the booking in the ✏️ settings and run the cell: `quote` prints the recipe, the intercept plus what each part of the booking adds, and the predicted rate. Hotels: `"Santiago"`, `"Viña del Mar"`, `"Puerto Varas"`. Rooms: `"Standard"`, `"Superior"`, `"Suite"`. Channels: `"Direct"`, `"Online travel agency"`, `"Corporate"`, `"Travel agent"`. Guests: `"Leisure"`, `"Business"`, `"Group"`.

In [3]:
# ✏️ Change these settings
HOTEL = "Puerto Varas"
MONTH = 7                      # 1 = January … 12 = December
ROOM = "Suite"
CHANNEL = "Online travel agency"
CUSTOMER = "Leisure"
NIGHTS, WEEKEND_NIGHTS = 4, 2
LEAD_TIME = 60                 # days between booking and arrival
ADULTS, CHILDREN = 2, 1

rate = quote(model, FEATURES, hotel=HOTEL, arrival_month=MONTH, room_type=ROOM, channel=CHANNEL, customer_type=CUSTOMER,
             nights=NIGHTS, weekend_nights=WEEKEND_NIGHTS, lead_time_days=LEAD_TIME, adults=ADULTS, children=CHILDREN)

                                                value  adds (USD/night)
start (intercept)  Santiago · Jan · Standard · Direct             132.8
hotel                                    Puerto Varas              41.0
arrival_month                                       7             -45.3
room_type                                       Suite              95.0
channel                          Online travel agency               5.5
customer_type                                 Leisure               0.0
nights                                              4             -13.1
weekend_nights                                      2               0.1
lead_time_days                                     60              -7.0
adults                                              2              18.0
children                                            1               8.7
PREDICTED RATE           USD 235.7 per night   (inputs left at their reference add 0)


**Think about it:** Which part of this booking adds the most to the rate? Now quote the **same booking in Santiago**: by how much does the rate change, and which line of the recipe is the only one that moved?

<details><summary>Show the answer</summary>

For the Puerto Varas suite in July, the room adds the most (about **+95**), followed by the hotel (**+41**). July itself *subtracts* about 45 against January. Moving the booking to Santiago changes **only the hotel line**: from +41 to 0, so the rate drops by about USD 41. Every other line stays exactly the same, because a linear model adds each part on its own.
</details>

The same recipe as a picture. The coefficients in the simulator are the ones your model just printed, rounded to one decimal.

<div class="hz-widget" data-widget="coef-explorer">
<p>▶ Interactive simulator: <a href="https://hizocar.github.io/mba-business-analytics-1/sessions/Session_3_Lab/">open this session on the course website</a> to use it.</p>
</div>

## <span class="hz-step-n">02</span> One change at a time

A revenue manager's favourite question: *"what if?"* Keep a booking fixed, change **one** input, and see how much the rate moves. In a linear model, the change is always **the coefficient** (for a category: the difference between two coefficients; for a number: the coefficient times the change).

In [4]:
# ✏️ Change these settings
BOOKING = dict(hotel="Viña del Mar", arrival_month=2, room_type="Standard", channel="Direct", nights=4, lead_time_days=30)
CHANGE_COLUMN = "room_type"    # try "channel", "nights", "arrival_month", "hotel", "lead_time_days"
NEW_VALUE = "Suite"            # e.g. "Corporate", 5, 7, "Santiago", 120

after_booking = {**BOOKING, CHANGE_COLUMN: NEW_VALUE}
before, after = quote(model, FEATURES, show=False, **BOOKING), quote(model, FEATURES, show=False, **after_booking)
part_before = contributions(model, FEATURES, BOOKING)[0][CHANGE_COLUMN]
part_after = contributions(model, FEATURES, after_booking)[0][CHANGE_COLUMN]
old = {**DEFAULTS, **BOOKING}[CHANGE_COLUMN]
print(f"Before: USD {before:.1f} per night  ({CHANGE_COLUMN} = {old})")
print(f"After:  USD {after:.1f} per night  ({CHANGE_COLUMN} = {NEW_VALUE})")
print(f"Change: USD {after - before:+.1f} = the {CHANGE_COLUMN} line going from {part_before:+.1f} to {part_after:+.1f}")

Before: USD 147.6 per night  (room_type = Standard)
After:  USD 242.6 per night  (room_type = Suite)
Change: USD +95.0 = the room_type line going from +0.0 to +95.0


**Think about it:** Try three changes and explain each one with a coefficient. (1) `channel` from Direct to `"Corporate"`. (2) `nights` from 4 to 5, then from 6 to 7: is the discount the same? (3) `hotel` to `"Santiago"` for this **February** booking: does the change make sense, given what the heatmap in the live session showed for Santiago in February?

<details><summary>Show the answer</summary>

(1) Corporate is about **USD 14 cheaper** than Direct: the `channel_Corporate` coefficient. (2) Every extra night lowers the rate by the **same** USD 3.3, whether it is the 5th or the 7th. Altamar's real discounts come in steps at 5 and 7 nights; a straight line smooths them out. (3) The model moves the booking down by about **USD 21** (Viña's +20.8 disappears), in any month. But Santiago is *quiet* in February, while Viña is at its peak: the real gap is much larger (about USD 105 per night in February, in the live session's chart of rates by month). The model gets it wrong because it has one February effect for all hotels.
</details>

## <span class="hz-step-n">03</span> The biggest misses

A typical error of USD 20 hides some very large ones. Find the test bookings where the model was most wrong and look for what they have in common. A positive `miss` means the guest paid **more** than predicted (the model would have underpriced the room).

In [5]:
# ✏️ Change these settings
TOP_N = 20                     # how many of the worst misses to show

view = test.assign(predicted=model.predict(X_test).round(1))
view["miss"] = (view["adr"] - view["predicted"]).round(1)
worst = view.loc[view["miss"].abs().sort_values(ascending=False).index].head(TOP_N)
print(worst[["hotel", "arrival_month", "room_type", "channel", "adr", "predicted", "miss"]].to_string(index=False))

       hotel  arrival_month room_type              channel    adr  predicted   miss
Viña del Mar              2     Suite Online travel agency 429.10      262.8  166.3
Puerto Varas              1     Suite Online travel agency 435.01      274.1  160.9
Puerto Varas              2     Suite Online travel agency 409.15      280.7  128.4
Viña del Mar              2     Suite               Direct 366.20      241.1  125.1
    Santiago              2     Suite               Direct 121.97      237.6 -115.6
Puerto Varas              2     Suite Online travel agency 400.25      285.7  114.6
    Santiago              2     Suite            Corporate  96.73      207.5 -110.8
    Santiago              1     Suite               Direct 125.41      234.8 -109.4
    Santiago              1     Suite Online travel agency 108.47      214.2 -105.7
    Santiago              1     Suite               Direct 119.63      223.8 -104.2
Puerto Varas              1     Suite Online travel agency 383.29      279.6

Compare the worst misses with the test set as a whole: how often are they summer arrivals (January or February), and how often suites?

In [6]:
summer = lambda d: d["arrival_month"].isin([1, 2]).mean()
suites = lambda d: (d["room_type"] == "Suite").mean()
print(f"January or February arrivals: {summer(worst):.0%} of the {TOP_N} worst misses, {summer(view):.0%} of all test bookings")
print(f"Suites:                       {suites(worst):.0%} of the {TOP_N} worst misses, {suites(view):.0%} of all test bookings")
print(f"Underpriced (miss > 0): {(worst['miss'] > 0).sum()} · overpriced (miss < 0): {(worst['miss'] < 0).sum()}")
print(worst.groupby(["hotel", "room_type"]).size().rename("worst misses").to_string())

January or February arrivals: 95% of the 20 worst misses, 26% of all test bookings
Suites:                       85% of the 20 worst misses, 10% of all test bookings
Underpriced (miss > 0): 11 · overpriced (miss < 0): 9
hotel         room_type
Puerto Varas  Suite        6
Santiago      Standard     1
              Suite        6
              Superior     2
Viña del Mar  Suite        5


**Think about it:** What do the worst misses have in common? Why does the model underprice some of them and overprice others? Try `TOP_N = 50`: does the pattern hold?

<details><summary>Show the answer</summary>

Almost all of them are **suites arriving in January or February**: peak summer. At **Viña del Mar and Puerto Varas** the model **underprices** them (guests paid USD 100 to 170 a night more than predicted), because suites go up *more* than other rooms in peak season, and the model only knows one fixed "+95" for a suite. At **Santiago** it **overprices** them, because it charges Santiago the January–February peak of the beach and the lake. Both are the same weakness: effects that depend on each other (room × season, hotel × month), which a model that adds one effect per input cannot draw.
</details>

<div class="hz-widget" data-widget="season-misfit">
<p>▶ Interactive simulator: <a href="https://hizocar.github.io/mba-business-analytics-1/sessions/Session_3_Lab/">open this session on the course website</a> to use it.</p>
</div>

## <span class="hz-step-n">04</span> Fewer inputs or more?

Every input costs something: data to collect, a column to explain to the revenue team. Is it worth it? Fit a model with only the inputs in ✏️ `FEATURES_SMALL` and compare it with the full model on the **same test set**. Available inputs: `hotel`, `arrival_month`, `room_type`, `channel`, `customer_type`, `nights`, `weekend_nights`, `lead_time_days`, `adults`, `children`, `deposit`, `repeat_guest`, `previous_cancellations`, `special_requests`, `parking`, `country`.

In [7]:
# ✏️ Change these settings
FEATURES_SMALL = ["hotel", "room_type"]

small = LinearRegression().fit(make_X(train, FEATURES_SMALL), y_train)
results = pd.DataFrame([
    scores(f"{len(FEATURES_SMALL)} inputs: " + ", ".join(FEATURES_SMALL), y_test, small.predict(make_X(test, FEATURES_SMALL))),
    scores(f"Full model: {len(FEATURES)} inputs", y_test, model.predict(X_test)),
])
print(results.to_string(index=False))

                     model  MAE (USD/night) MAE as % of avg rate    R²
2 inputs: hotel, room_type             29.8                  23% 0.505
     Full model: 16 inputs             20.5                  16% 0.738


**Think about it:** Start from `["hotel", "room_type"]` and add one input at a time. Which single input brings the biggest drop in MAE? Then try the full list **without** `"country"`, `"deposit"` and `"special_requests"`: does the model get worse?

<details><summary>Show the answer</summary>

Hotel and room alone give a typical miss of about **USD 30** (R² ≈ 0.50). Adding **`arrival_month`** is by far the biggest single step: about **USD 23** (R² ≈ 0.68); no other input moves the MAE by more than a dollar on its own. `channel` and the guest counts then bring it down to about USD 21, and the rest to 20.5. Removing `country`, `deposit` and `special_requests` changes the MAE by only a few cents: they carry almost no information about the **rate** (they matter more for cancellations, next week). A simpler model that is just as accurate is easier to explain and to trust.
</details>

## <span class="hz-step-n">05</span> A note to the revenue manager

The revenue manager will not read this notebook. They need **two sentences**: what the model is good for, and where not to trust it. The cell below prints the numbers you need.

In [8]:
mae = mean_absolute_error(y_test, view["predicted"])
cells = view.pivot_table(index="hotel", columns="arrival_month", values="miss", aggfunc="mean")
hotel_w, month_w = cells.abs().stack().idxmax()
print(f"Typical error: USD {mae:.0f} per night, {mae / y_test.mean():.0%} of the average rate · test R² {r2_score(y_test, view['predicted']):.2f}")
print(f"Typical error with each hotel's average rate instead: USD {mean_absolute_error(y_test, test['hotel'].map(train.groupby('hotel')['adr'].mean())):.0f}")
print(f"Typical error on suites: USD {view.loc[view['room_type'] == 'Suite', 'miss'].abs().mean():.0f} · on Standard rooms: USD {view.loc[view['room_type'] == 'Standard', 'miss'].abs().mean():.0f}")
print(f"Worst hotel-month: {hotel_w} in {MONTHS[month_w - 1]}, average miss USD {cells.loc[hotel_w, month_w]:+.0f} per night")

Typical error: USD 20 per night, 16% of the average rate · test R² 0.74
Typical error with each hotel's average rate instead: USD 36
Typical error on suites: USD 47 · on Standard rooms: USD 17
Worst hotel-month: Santiago in Jan, average miss USD -74 per night


Write your note in the ✏️ cell (keep the quotes) and run it.

In [9]:
# ✏️ Change these settings
NOTE = """
Good for: ...
Do not trust it for: ...
"""
print(NOTE)


Good for: ...
Do not trust it for: ...



<details><summary>Show an example note</summary>

*"The model suggests a starting rate for Standard and Superior rooms outside the summer peak that is typically within USD 20 of what guests pay, almost twice as accurate as quoting each hotel's average; use it to set a first quote and to flag quotes that look far off. Do not use it on its own for suites in January–February, or for Santiago in the summer and Puerto Varas in July, where it is off by USD 30 to 70 a night in a predictable direction, until we fit a model that learns each hotel's seasons (week 5)."*

Notice: one use, one number, one comparison, and a limit that names **where**.
</details>

## Graded quiz

Five questions on this week's ideas. They count for **5% of the module grade**, and **only your first submission counts**, so finish the lab before you start. Use the name and email you are registered with in the program.

<div class="hz-widget" data-widget="graded-quiz">
<script type="application/json">{"id": "session-3", "title": "Week 3 quiz", "questions": [{"q": "On the test set the model's MAE is USD 20 and the average rate is USD 127. Which sentence should go to the general manager?", "options": ["The model is right 84% of the time.", "On new bookings, the model's rate is typically off by about USD 20 per night, around 16% of the average rate.", "The model explains 20% of the differences between rates.", "Over a whole stay, the model is off by USD 20 in total."]}, {"q": "With Standard as the reference room, the coefficients are Superior +31 and Suite +95. Everything else equal, how much more does the model charge for a Suite than for a Superior room?", "options": ["USD 95", "USD 126", "USD 64", "USD 31"]}, {"q": "A new model scores R² 0.99 on the training set and 0.41 on the test set. What is the right reading?", "options": ["It memorized the training bookings (overfitting); judge it by the test score, which is worse than our linear model's 0.74.", "It is the best model so far, because 0.99 is close to perfect.", "The test set must contain errors, so it should be refitted on all the data and scored again.", "The two numbers average to 0.70, about the same as our linear model."]}, {"q": "The worst misses are suites in January and February at Viña del Mar and Puerto Varas, all underpriced by USD 100 or more. What is the most useful next step?", "options": ["Score the model on the training set instead, where these misses are smaller.", "Add booking_id as an input, so that the model can recognise those bookings.", "Delete suites from the data, so that the model looks more accurate.", "Use a model that can learn effects that depend on each other (room × season, hotel × month), such as the trees of week 5."]}, {"q": "The coefficient of weekend_nights is almost 0. A manager concludes: “weekends do not affect our prices.” What is the best reply?", "options": ["Weekends do matter, but only for cancellations, never for rates.", "The coefficient is near zero only because weekend_nights is measured in nights instead of days.", "Not necessarily: weekends are cheaper in Santiago and dearer in Viña del Mar, and one coefficient for all hotels can average the two away.", "Correct: a coefficient near zero proves that the input has no effect."]}]}</script>
</div>